# 05_训练小分类器：练习参考答案

先完成同名课程再看。下面的初始化不依赖上一个Notebook内存。


In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'tools/learning.py').is_file() and (p / 'config.json').is_file())
sys.path.insert(0, str(ROOT / 'tools'))
import learning as L
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)


使用restored预测前两个样本，沿类别维选最大分数，得到长度2的类别编号。


构造可学习的教学数据


In [ ]:
get_ipython().run_line_magic('matplotlib', 'inline')
import torch
from torch import nn
import matplotlib.pyplot as plt
torch.set_num_threads(4)
torch.manual_seed(42)
y = torch.arange(40).repeat(4)
x = torch.nn.functional.one_hot(y, 40).float() + 0.03 * torch.randn(len(y), 40)
model = nn.Linear(40, 40)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.1)
print(x.shape, y.shape, model(x).shape)


一次循环中的先后顺序


In [ ]:
losses = []
before = model.weight.detach().clone()
model.train()
for step in range(60):
    optimizer.zero_grad()
    logits = model(x)
    loss = nn.functional.cross_entropy(logits, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
assert not torch.equal(before, model.weight)
print('首末loss', round(losses[0], 4), round(losses[-1], 4))
plt.plot(losses)
plt.xlabel('step'); plt.ylabel('training loss'); plt.show()


新对象加载本地参数


In [ ]:
model.eval()
with torch.inference_mode():
    expected = model(x[:2]).clone()
weight_path = OUT / 'toy-classifier.pt'
torch.save(model.state_dict(), weight_path)
restored = nn.Linear(40, 40)
restored.load_state_dict(torch.load(weight_path, map_location='cpu', weights_only=True))
restored.eval()
with torch.inference_mode():
    torch.testing.assert_close(restored(x[:2]), expected)
print('重载前后输出一致；仅教学模型：', weight_path)


In [ ]:
import torch
from torch import nn
restored = nn.Linear(40, 40)
restored.load_state_dict(torch.load(OUT / 'toy-classifier.pt', weights_only=True))
restored.eval()
x = torch.eye(40)[:2]
with torch.inference_mode():
    your_answer = restored(x).argmax(dim=1)
assert your_answer.tolist() == [0, 1]
print(your_answer)


## 为什么这样写

state_dict是参数而非整个训练进程；需建立相同结构并load。参考答案读取本课的教学权重，独立验收前会先生成该教学文件。

逐句解释训练循环；重启后能加载文件；知道训练loss下降不保证泛化。
